# Explainable AML monitoring: run the pipeline in order

This notebook walks through the code one section at a time. The saved **IBM data profile** output is a snapshot from the full downloaded CSV. The feature and model teaching sections use a **small artificial demo**. The final section reads a separate **10% sampled IBM pilot** with real labels and complete prior transaction history; it is not a full-dataset experiment.

Follow the [README setup and data-download steps](../README.md) before a fresh **Run All**. Select this repository's `.venv/bin/python` kernel. Without local generated results, the IBM profile and sampled-study cells display reproduction commands instead of tables. For the source-code reading order, see [`docs/START_HERE.md`](../docs/START_HERE.md).

## What this notebook covers

Sections 0–7 run the artificial demo step by step. Section 1 also reads a saved profile of the full IBM CSV. Sections 8–9 display model results and explanations from the separate 10% sampled IBM study. The demo metrics and predictions are displayed here and kept in memory, with no extra demo files. A fresh run of the real-data sections needs the [README setup and data commands](../README.md).

The [MVP scope](../docs/PROJECT_SCOPE.md) explains what belongs in the class project.


## 0. Setup

Find the project root, check that VS Code selected this repository's `.venv` kernel and required packages, then import the same functions used by the CLI. The first code cell stops with instructions if another environment is selected.

In [1]:
from pathlib import Path
import importlib.util
import json
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the AML project folder")

PROJECT_ENV = PROJECT_ROOT / ".venv"
if Path(sys.prefix).resolve() != PROJECT_ENV.resolve():
    raise RuntimeError(
        f"Wrong notebook kernel: {sys.executable}\n"
        "In VS Code, click the kernel name at the top right, choose "
        "Select Another Kernel -> Python Environments, then select "
        f"{PROJECT_ENV / 'bin' / 'python'}. Restart the kernel and Run All."
    )

required = ("numpy", "pandas", "sklearn", "xgboost", "interpret", "IPython")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        f"Missing packages in the project environment: {missing}. "
        "From the project folder, run: .venv/bin/python -m pip install -e '.[dev,notebook]'"
    )

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from aml_monitoring.data import make_demo_transactions
from aml_monitoring.evaluation import chronological_split, score_predictions
from aml_monitoring.features import FEATURES, build_features
from aml_monitoring.models import fit_model, rule_scores

SEED = 42
DEMO_ROWS = 1200
ALERT_CAPACITY = 25
print("Kernel: project .venv")
print(f"Project: {PROJECT_ROOT.name}")
print(f"Demo: {DEMO_ROWS:,} artificial rows; alert capacity: {ALERT_CAPACITY}")

Kernel: project .venv
Project: Explainable-AML-Transaction-Monitoring
Demo: 1,200 artificial rows; alert capacity: 25


## 1. Inspect the real IBM data profile

The profile was produced by `.venv/bin/aml --csv data/raw/HI-Small_Trans.csv --profile-only --output results/ibm`. This cell reads the **locally generated** JSON instead of loading 5 million rows again. The JSON is ignored by Git; a fresh clone must download and profile the CSV first. The saved output below is a snapshot of that run.

The late-period label shift motivates the study's separate stress period and limits broader claims.

In [2]:
profile_path = PROJECT_ROOT / "results" / "ibm" / "data_profile.json"
if profile_path.exists():
    ibm_profile = json.loads(profile_path.read_text())
    display(pd.DataFrame([{
        "transactions": ibm_profile["rows"],
        "labeled_laundering": ibm_profile["positives"],
        "label_rate_pct": round(100 * ibm_profile["positive_rate"], 4),
        "first_timestamp": ibm_profile["first_timestamp"],
        "last_timestamp": ibm_profile["last_timestamp"],
    }]))
    split_profile = pd.DataFrame(ibm_profile["splits"]).T
    display(split_profile[["rows", "positives", "positive_rate", "first_timestamp", "last_timestamp"]])
    daily_profile = pd.DataFrame(ibm_profile["daily_activity"]).T
    daily_profile["label_rate_pct"] = 100 * daily_profile["positives"] / daily_profile["rows"]
    print("Last eight days of the IBM file:")
    display(daily_profile.tail(8))
else:
    ibm_profile = None
    print("IBM profile not found. Run the profile command in the text above, then rerun this cell.")

,transactions,labeled_laundering,label_rate_pct,first_timestamp,last_timestamp
0,5078345,5177,0.1019,2022-09-01 00:00:00,2022-09-18 16:18:00


,rows,positives,positive_rate,first_timestamp,last_timestamp
train,3047216,2299,0.000754,2022-09-01 00:00:00,2022-09-06 13:36:00
validation,1015565,1081,0.001064,2022-09-06 13:37:00,2022-09-08 16:12:00
test,1015564,1797,0.001769,2022-09-08 16:13:00,2022-09-18 16:18:00


Last eight days of the IBM file:


,rows,positives,label_rate_pct
2022-09-11,396,232,58.585859
2022-09-12,281,170,60.498221
2022-09-13,184,106,57.608696
2022-09-14,121,70,57.851240
2022-09-15,46,28,60.869565
2022-09-16,46,26,56.521739
2022-09-17,23,15,65.217391
2022-09-18,11,8,72.727273


## 2. Load example transactions (`data.py`)

`make_demo_transactions()` gives us a small, reproducible dataset in the IBM column format. Its labels are deliberately easy to learn so this section checks the code path, not research performance.

In [3]:
transactions = make_demo_transactions(n=DEMO_ROWS, seed=SEED)
print(f"Rows: {len(transactions):,}; labeled positives: {transactions['is_laundering'].sum():,}")
display(transactions[[
    "timestamp", "from_bank", "from_account", "to_bank", "to_account",
    "amount_paid", "payment_currency", "payment_format", "is_laundering",
]].head(5))

Rows: 1,200; labeled positives: 87


,timestamp,from_bank,from_account,to_bank,to_account,amount_paid,payment_currency,payment_format,is_laundering
0,2024-01-01 00:00:00,002,S0004,001,R0048,605.380337,USD,ACH,0
1,2024-01-01 00:30:00,002,S0034,003,R0026,425.069330,USD,ACH,0
2,2024-01-01 01:00:00,002,S0029,001,R0054,2507.515129,USD,ACH,0
3,2024-01-01 01:30:00,002,S0019,003,R0041,280.541117,USD,ACH,0
4,2024-01-01 02:00:00,002,S0019,003,R0035,10817.162278,USD,Wire,1


## 3. Split by time (`evaluation.py`)

Training uses the earliest rows, validation the middle rows, and test the latest rows. Rows with an identical timestamp stay together. The test labels are shown here for learning purposes, but the models fit only on training labels.

In [4]:
split = chronological_split(transactions["timestamp"])
split_summary = pd.DataFrame([
    {
        "period": period,
        "rows": len(indexes),
        "positives": int(transactions.iloc[indexes]["is_laundering"].sum()),
        "first": transactions.iloc[indexes]["timestamp"].min(),
        "last": transactions.iloc[indexes]["timestamp"].max(),
    }
    for period, indexes in split.items()
])
display(split_summary)

,period,rows,positives,first,last
0,train,720,44,2024-01-01,2024-01-15 23:30:00
1,validation,240,21,2024-01-16,2024-01-20 23:30:00
2,test,240,22,2024-01-21,2024-01-25 23:30:00


## 4. Build point-in-time features (`features.py`)

Each row gets current-transaction fields plus the sender's earlier 24-hour and 7-day activity. Transactions at the same time cannot see one another. We build the chronological history across periods because earlier transactions are known when later ones are scored; labels are never used in feature construction.

In [5]:
features = build_features(transactions)
print(f"Feature matrix: {features.shape[0]:,} rows × {features.shape[1]} columns")
display(features[[
    "log_amount_paid", "prior_txn_count_24h", "prior_amount_24h",
    "prior_txn_count_7d", "prior_avg_amount_7d", "new_counterparty_7d",
    "payment_format",
]].head(5))
prior_rows = features.index[features["prior_txn_count_24h"] > 0]
if len(prior_rows):
    row = int(prior_rows[0])
    print(f"Example row with prior activity: {row}")
    display(features.loc[[row], FEATURES])

Feature matrix: 1,200 rows × 13 columns


,log_amount_paid,prior_txn_count_24h,prior_amount_24h,prior_txn_count_7d,prior_avg_amount_7d,new_counterparty_7d,payment_format
0,6.407507,0,0.000000,0,0.000000,1,ACH
1,6.054602,0,0.000000,0,0.000000,1,ACH
2,7.827446,0,0.000000,0,0.000000,1,ACH
3,5.640279,0,0.000000,0,0.000000,1,ACH
4,9.288982,1,280.541117,1,280.541117,1,Wire


Example row with prior activity: 4


,log_amount_paid,hour,day_of_week,same_bank,prior_txn_count_24h,prior_amount_24h,prior_txn_count_7d,prior_avg_amount_7d,amount_vs_prior_avg_7d,new_counterparty_7d,payment_format,payment_currency,receiving_currency
4,9.288982,2,0,0,1,280.541117,1,280.541117,38.558206,1,Wire,USD,USD


## 5. Fit the four baselines (`models.py`)

The fixed rules require no fitting. Logistic regression, XGBoost, and EBM fit on **training rows only**. Categorical values are encoded inside the logistic and XGBoost pipelines; EBM reads the named columns directly.

In [6]:
labels = transactions["is_laundering"].to_numpy()
trained = {}
for name in ("logistic", "xgboost", "ebm"):
    trained[name] = fit_model(name, features.iloc[split["train"]][FEATURES],
                              labels[split["train"]], seed=SEED)
    print(f"Fitted {name} on {len(split['train']):,} training rows")
print("Rules: fixed thresholds; no training")

Fitted logistic on 720 training rows
Fitted xgboost on 720 training rows


Fitted ebm on 720 training rows
Rules: fixed thresholds; no training


## 6. Score and evaluate (`evaluation.py`)

We compare validation and test ranking quality and how many positive labels appear within the highest-scored `K` alerts. The `pr_auc` JSON field is calculated with scikit-learn's **average precision** function, a summary of the precision–recall curve. `recall_at_k` measures detections at a fixed review capacity. Demo figures below have no research meaning.

In [7]:
metrics = {}
prediction_parts = []
for name in ("rules", "logistic", "xgboost", "ebm"):
    metrics[name] = {}
    for period in ("validation", "test"):
        indexes = split[period]
        x = features.iloc[indexes][FEATURES]
        scores = rule_scores(x) if name == "rules" else trained[name].predict_proba(x)[:, 1]
        metrics[name][period] = score_predictions(labels[indexes], scores, ALERT_CAPACITY)
        prediction_parts.append(pd.DataFrame({
            "source_row": indexes,
            "timestamp": transactions.iloc[indexes]["timestamp"].to_numpy(),
            "period": period,
            "model": name,
            "label": labels[indexes],
            "score": scores,
        }))
predictions = pd.concat(prediction_parts, ignore_index=True)
comparison = pd.DataFrame([
    {"model": model, "period": period, **values}
    for model, periods in metrics.items()
    for period, values in periods.items()
])
display(comparison[[
    "model", "period", "pr_auc", "roc_auc", "precision_at_k",
    "recall_at_k", "positives_found", "alerts",
]].round(4))

,model,period,pr_auc,roc_auc,precision_at_k,recall_at_k,positives_found,alerts
0,rules,validation,0.8248,0.9631,0.72,0.8571,18,25
1,rules,test,0.8408,0.9656,0.68,0.7727,17,25
2,logistic,validation,1.0000,1.0000,0.84,1.0000,21,25
3,logistic,test,1.0000,1.0000,0.88,1.0000,22,25
4,xgboost,validation,1.0000,1.0000,0.84,1.0000,21,25
5,xgboost,test,1.0000,1.0000,0.88,1.0000,22,25
6,ebm,validation,1.0000,1.0000,0.84,1.0000,21,25
7,ebm,test,1.0000,1.0000,0.88,1.0000,22,25


## 7. Look at the highest-ranked demo transactions

The `predictions` DataFrame holds every demo score in memory. The table below shows the highest-ranked XGBoost test transactions beside their source fields. These demo scores are **not explanations**; the separate IBM study exports Tree SHAP examples and EBM global terms. A high score ranks a transaction for review; it does not prove laundering.


In [8]:
top = predictions.query("model == 'xgboost' and period == 'test'").nlargest(5, "score")
examples = transactions.loc[top["source_row"], [
    "timestamp", "from_account", "to_account", "amount_paid",
    "payment_currency", "payment_format", "is_laundering",
]].copy()
examples["xgboost_score"] = top["score"].to_numpy()
display(examples)

,timestamp,from_account,to_account,amount_paid,payment_currency,payment_format,is_laundering,xgboost_score
1027,2024-01-22 09:30:00,S0012,R0051,18628.863323,USD,Wire,1,0.998334
1028,2024-01-22 10:00:00,S0025,R0015,12344.826719,USD,Wire,1,0.998334
1058,2024-01-23 01:00:00,S0019,R0000,29615.678908,USD,Wire,1,0.998334
1075,2024-01-23 09:30:00,S0007,R0023,27588.505206,USD,Wire,1,0.998334
1121,2024-01-24 08:30:00,S0003,R0001,18362.742561,USD,Wire,1,0.998334


## From demo to the sampled IBM pilot

The full IBM profile shows a sharp shift after September 10: only 1,108 later transactions, 655 of them positive. The pilot uses September 1–10 for its primary comparison, keeps the late period separate, and samples 10% of primary rows for model fitting and evaluation. DuckDB computes behavioral history from **all** earlier original transactions before those modeling rows are selected. See the [study report](../docs/STUDY_RESULTS.md) before interpreting any score.

## 8. Inspect the sampled real-data study

This cell displays the separate study's local model comparison and diagnostics. Its saved output is a snapshot. To regenerate it, follow [Run the sampled IBM pilot](../README.md#run-the-sampled-ibm-pilot). The study reads the full CSV to compute earlier history, then fits and evaluates on a 10% sample.


In [9]:
study_dir = PROJECT_ROOT / "results" / "study_full_history_10pct"
if (study_dir / "metrics.json").exists():
    study_manifest = json.loads((study_dir / "study_manifest.json").read_text())
    study_metrics = json.loads((study_dir / "metrics.json").read_text())
    chosen = study_manifest["selected_by_validation_average_precision"]
    print(f"Sample: {study_manifest['sample_fraction_primary']:.0%} of the primary period; "
          f"{study_manifest['periods']['test']['rows']:,} primary test rows and "
          f"{study_manifest['periods']['test']['positives']} positive labels.")
    print(f"Validation-selected model: {chosen}")
    study_comparison = pd.DataFrame([
        {
            "model": name,
            "validation_AP": values["validation"]["pr_auc"],
            "test_AP": values["test"]["pr_auc"],
            "test_precision_at_100": values["test"]["precision_at_k"],
            "test_recall_at_100": values["test"]["recall_at_k"],
        }
        for name, values in study_metrics.items()
    ])
    display(study_comparison.round(4))
    diagnostics_path = study_dir / "diagnostics.json"
    if diagnostics_path.exists():
        study_diagnostics = json.loads(diagnostics_path.read_text())
        low, high = study_diagnostics["stratified_bootstrap_95pct_interval"]
        print(f"Descriptive bootstrap AP interval: {low:.4f}–{high:.4f}")
    alerts_path = study_dir / "top_100_alerts.csv"
    if alerts_path.exists():
        print("Highest-ranked primary-test alerts:")
        display(pd.read_csv(alerts_path).head(10))
    print("Study design and interpretation: docs/STUDY_RESULTS.md")
else:
    print("Sampled study outputs not found. Run the study command in the section above.")

Sample: 10% of the primary period; 86,066 primary test rows and 103 positive labels.
Validation-selected model: xgboost_behavioral


,model,validation_AP,test_AP,test_precision_at_100,test_recall_at_100
0,rules_behavioral,0.0010,0.0011,0.00,0.0000
1,logistic_transaction,0.0120,0.0135,0.01,0.0097
2,xgboost_transaction,0.0336,0.0459,0.12,0.1165
3,ebm_transaction,0.0185,0.0250,0.01,0.0097
4,logistic_behavioral,0.0133,0.0111,0.00,0.0000
5,xgboost_behavioral,0.3362,0.2245,0.23,0.2233
6,ebm_behavioral,0.1519,0.1105,0.18,0.1748


Descriptive bootstrap AP interval: 0.1468–0.3128
Highest-ranked primary-test alerts:


,source_row,timestamp,period,model,label,score,payment_currency,alert_rank
0,3459535,2022-09-10 11:13:00,test,xgboost_behavioral,1,0.997553,US Dollar,1
1,4887997,2022-09-10 18:39:00,test,xgboost_behavioral,1,0.997352,US Dollar,2
2,4683032,2022-09-10 11:54:00,test,xgboost_behavioral,1,0.997142,Euro,3
3,3117691,2022-09-10 08:39:00,test,xgboost_behavioral,1,0.996361,Euro,4
4,4974359,2022-09-10 10:51:00,test,xgboost_behavioral,1,0.995921,Euro,5
5,3961342,2022-09-09 08:30:00,test,xgboost_behavioral,1,0.995544,US Dollar,6
6,3249689,2022-09-10 17:11:00,test,xgboost_behavioral,1,0.995426,US Dollar,7
7,4683036,2022-09-10 11:16:00,test,xgboost_behavioral,1,0.995154,Yuan,8
8,3303613,2022-09-09 12:19:00,test,xgboost_behavioral,1,0.994368,Euro,9
9,4389507,2022-09-09 12:54:00,test,xgboost_behavioral,1,0.993482,Australian Dollar,10


Study design and interpretation: docs/STUDY_RESULTS.md


## 9. Compare explanations

The study exports XGBoost feature importance, one transaction's Tree SHAP contributions, and EBM global term importance. These describe how the fitted models scored synthetic transactions; they are not evidence of real laundering. The local SHAP values are on the model's raw log-odds scale.


In [10]:
explanations_path = study_dir / "explanations.json"
if explanations_path.exists():
    explanations = json.loads(explanations_path.read_text())
    xgb = explanations["xgboost_behavioral"]
    ebm = explanations["ebm_behavioral"]
    print("XGBoost's leading features:")
    display(pd.DataFrame(xgb["top_importances"]).head(8).round(4))
    print("EBM's leading global terms:")
    display(pd.DataFrame(ebm["top_terms"]).head(8).round(4))
    local = xgb["local_tree_shap_log_odds"][0]
    print(f"Tree SHAP contributions for high-ranked source row {local['source_row']}:")
    display(pd.DataFrame(local["top_terms"]).round(4))
else:
    print("Study explanations not found. Run the sampled study command in the README.")

XGBoost's leading features:


,feature,importance
0,categorical__payment_format_ACH,0.2377
1,categorical__payment_format_Reinvestment,0.0989
2,numeric__new_counterparty_7d,0.0527
3,categorical__payment_format_Cheque,0.0501
4,numeric__same_bank,0.0497
5,numeric__prior_txn_count_7d,0.0468
6,categorical__payment_format_Credit Card,0.0467
7,categorical__payment_currency_Saudi Riyal,0.0343


EBM's leading global terms:


,term,importance
0,payment_format,2.4242
1,new_counterparty_7d,1.1540
2,prior_txn_count_7d,0.5504
3,amount_vs_prior_avg_7d,0.4765
4,prior_txn_count_24h,0.4481
5,log_amount_paid,0.4405
6,prior_amount_24h,0.3268
7,day_of_week,0.2699


Tree SHAP contributions for high-ranked source row 3459535:


,feature,contribution
0,categorical__payment_format_ACH,2.7552
1,numeric__new_counterparty_7d,0.6946
2,numeric__prior_avg_amount_7d,0.5402
3,numeric__amount_vs_prior_avg_7d,0.4398
4,numeric__log_amount_paid,0.3628
5,numeric__prior_amount_24h,0.2728
6,numeric__day_of_week,0.1825
7,categorical__payment_format_Credit Card,0.0598
8,categorical__payment_format_Cheque,0.0529
9,categorical__payment_format_Reinvestment,0.0428


## Before the final class report

The sampled pilot has complete prior history for selected rows. For a full-data claim, fit and evaluate on all eligible primary-period rows, investigate the late-period shift, repeat the comparison across seeds or windows, and review case-level explanations. See the [MVP scope](../docs/PROJECT_SCOPE.md).
